Apply the below transformation

- Create a misc transactions by excluding Sell and Buy Trans_Code, add new columns for processing date/month/year 
- Modify the Transaction amount clean column to show the valid field to consider for calculations
- Split the Multiline Description to get the details to identify the Stock_ETF, CUSIP
- Create a new table as valid_transactions and miscellaneous transaction
- Create a new table from statement current portfolio section - Rename the column names and round the amount fields and price for float of 2 position   

In [0]:
%sql
select * from investment_vision.history_table_january;

In [0]:
%python
from pyspark.sql.functions import *
# inv_types_df = spark.sql("select * from investment_vision.raw_invest_types")
misc1_df = spark.sql(
    """
    SELECT * from investment_vision.history_table_january
    WHERE Trans_Code NOT IN ('Sell', 'Buy', "CDIV")
    """
)

display(misc1_df)


In [0]:
%python
from pyspark.sql.functions import *
display(misc1_df)
misc1_df.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable(
    'workspace.investment_vision.misc_transactions')

In [0]:
%python
from pyspark.sql.functions import *
# inv_types_df = spark.sql("select * from investment_vision.raw_invest_types")
valid1_df = spark.sql(
    """
    SELECT * from investment_vision.history_table_january
    WHERE Trans_Code IN ('Sell', 'Buy', "CDIV")
    """
)
display(valid1_df)


In [0]:
from pyspark.sql.functions import *

valid1_df = (
    valid1_df
    .withColumn(
        "Transaction_Amount_Clean",
        abs(
            when(
                col("Amount") == "1",
                (col("price").cast("decimal(18,2)") * col("quantity").cast("decimal(18,2)"))
            ).when(
                col("Amount").rlike(r"^\(\$"),
                -1 * (
                    regexp_replace(
                        regexp_replace(col("Amount"), r"[\(\)\$]", ""), ",", ""
                    ).cast("decimal(18,2)")
                )
            ).otherwise(
                regexp_replace(
                    regexp_replace(col("Amount"), r"[\$]", ""), ",", ""
                ).cast("decimal(18,2)")
            )
        )
    )
    # .dropDuplicates()
)
display(valid1_df)


In [0]:
%python
from pyspark.sql.functions import *
display(valid1_df)
valid1_df.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable('workspace.investment_vision.valid_transactions')

In [0]:
%sql
select * from investment_vision.raw_portfolio_statement

In [0]:
from pyspark.sql.functions import *

portfolio_df = spark.table('workspace.investment_vision.raw_portfolio_statement')

agg_df = portfolio_df.groupBy('ticker').agg(
    round(avg(regexp_replace(col('price'), ',', '').cast('double')), 2).alias('current_price'),
    round(sum(regexp_replace(col('qty'), ',', '').cast('double')), 2).alias('total_qty'),
    round(sum(regexp_replace(col('market_value'), ',', '').cast('double')), 2).alias('current_market_value')
)

display(agg_df)

agg_df.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable('workspace.investment_vision.current_portfolio_value')
